In [1]:
import numpy as np
import pandas as pd
import polars as pl

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from xgboost import XGBRegressor


In [2]:

df_pl = pl.read_parquet("../data/processed/stores_sales_features.parquet")
df = df_pl.to_pandas()

### 1. Setting target and features

In [3]:
df = df.sort_values("order_date").reset_index(drop=True)
df["log_sales"] = np.log1p(df["sales"])

target = "log_sales"
cat = ["segment","region","category","sub_category","ship_mode"]
num = ["quantity","discount","order_year","order_quarter","order_month","order_weekday","order_weekend"]

### 2. Train, test

In [4]:
train = df[df["order_year"] < 2017]
test  = df[df["order_year"] == 2017]

X_train = train[cat + num]
y_train = train[target]            # already log-transformed
X_test  = test[cat + num]
y_test  = test["sales"]            # evaluate on real scale

### 3. Preprocess

In [5]:
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ("num", "passthrough", num)
])

### 4. Linear Regression

In [6]:
# Linear Regression baseline
model = Pipeline([
    ("prep", preprocess),
    ("lr", LinearRegression())
])

model.fit(X_train, y_train)
pred_log = model.predict(X_test)
pred = np.expm1(pred_log)

# Saving results
lr_mae = mean_absolute_error(y_test, pred)
lr_rmse = root_mean_squared_error(y_test, pred)
lr_r2 = r2_score(y_test, pred)


### 5. Random Forest

In [7]:
# Random Forest
rf = Pipeline([
    ("prep", preprocess),
    ("rf", RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_split=2,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)           # <--- FIXED
pred_log = rf.predict(X_test)
pred = np.expm1(pred_log)

# Saving results
rf_mae = mean_absolute_error(y_test, pred)
rf_rmse = root_mean_squared_error(y_test, pred)
rf_r2 = r2_score(y_test, pred)


### 6. XGBoost

In [8]:

xgb = Pipeline([
    ("prep", preprocess),
    ("xgb", XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=4,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

xgb.fit(X_train, y_train)  # train on log_sales
pred_log = xgb.predict(X_test)
pred = np.expm1(pred_log)

# Saving results
xgb_mae = mean_absolute_error(y_test, pred)
xgb_rmse = root_mean_squared_error(y_test, pred)
xgb_r2 = r2_score(y_test, pred)


### Saving all results

In [11]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest", "XGBoost"],
    "MAE": [lr_mae, rf_mae, xgb_mae],
    "RMSE": [lr_rmse, rf_rmse, xgb_rmse],
    "R2": [lr_r2, rf_r2, xgb_r2]
})
results

,Model,MAE,RMSE,R2
0,Linear Regression,174.013579,353.290021,0.387317
1,Random Forest,173.746349,339.737870,0.433420
2,XGBoost,166.912830,325.397666,0.480241


### Feature Importance

In [12]:
importances = xgb.named_steps["xgb"].feature_importances_
feature_names = xgb.named_steps["prep"].get_feature_names_out()

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

fi.head(15)


,feature,importance
10,cat__sub_category_Furnishings,0.467623
8,cat__sub_category_Bookcases,0.096271
11,cat__sub_category_Tables,0.092608
9,cat__sub_category_Chairs,0.056045
16,num__quantity,0.047723
17,num__discount,0.025417
14,cat__ship_mode_Second Class,0.015213
5,cat__region_South,0.014922
0,cat__segment_Consumer,0.014198
1,cat__segment_Corporate,0.014117
